In [50]:
import os
import pandas as pd
import re
import torch
from collections import defaultdict
from pathlib import Path
import numpy as np

def get_indiv_concepts(formula) -> set:
    concepts = set()
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.add(c[:end_idx])
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    raw=[]
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
        raw.extend(concepts)
    
    return unit_concepts, set(raw)

def build_binary_mask(neuron_mask, foundational_concept_list) -> torch.Tensor:
    num_neurons = len(neuron_mask)
    num_concepts = len(foundational_concept_list)

    # Step 1: Initialize tensor
    tensor = torch.zeros((num_neurons, num_concepts), dtype=torch.float32)

    # Step 2: Fill in ones
    for i, concepts in enumerate(neuron_mask.values()):
        for j, concept in enumerate(foundational_concept_list):
            if concept in concepts:
                tensor[i, j] = 1.0

    # Step 3: Compute row sums
    row_sums = tensor.sum(dim=1, keepdim=True)

    # Step 4: Normalize safely
    dist_tensor = torch.zeros_like(tensor)
    row_mask = (row_sums != 0).squeeze(1)  # True for rows with sum > 0
    dist_tensor[row_mask] = tensor[row_mask] #/ row_sums[row_mask]

    return dist_tensor



def get_neurons_for_cps(concepts, mapping):
    neurons = []
    for neuron, cps in mapping.items():
        for c in cps:
            if c in concepts:
                neurons.append(neuron)
                break
    return neurons

def get_all_cps_for_pi(folder):
    root_path = Path(folder)

    # Find all matching CSV files
    csv_pattern = 'Cluster*IOUS1024N.csv'
    csv_files = list(root_path.rglob(csv_pattern))
    
    concept_dict=defaultdict(set)
    s=set()
    
    for csv_file in csv_files:
        concepts = []
        csv_file = os.path.join(folder, csv_file)
        df = pd.read_csv(csv_file)
        for unit, formula in zip(df.unit, df.best_name):
            concept_dict[unit].update(get_indiv_concepts(formula))
            all_concepts.extend(get_indiv_concepts(formula))
        
     
        for u, c in concept_dict.items():
            s.update(c)
   
    return concept_dict
def get_all_cps_for(folder, cluster):
    root_path = Path(folder)

    
    s=set()
    
    all_concepts = []
    
    csv_file = os.path.join(folder, f'Cluster{cluster}IOUS1024N.csv')
    df = pd.read_csv(csv_file)
    for unit, formula in zip(df.unit, df.best_name):
        for cp in get_indiv_concepts(formula):
            base = cp.split(":")[-1]
            all_concepts.append(base)
            s.add(base)
    
    return all_concepts, s

In [53]:
for model in ['LLAMA', 'BERT'] :
    for method in ['CoFi', 'wanda', 'lottery_ticket'] :
        for file in ['Run0.25_5']:
            folder = f"/workspace/CCE_NLI/{model}/exp/{method}/{file}/Expls"
            if not os.path.exists(folder): continue
            for pi in os.listdir(folder):
                try:
                    for cluster in range(1,4):
                        all_cps, setofcp =  get_all_cps_for(os.path.join(folder, pi), cluster)
            
                        print(f"Unique for {model}, {method} {pi} {cluster}: ", 100*len(setofcp)/len(all_cps))
                except Exception as e:
                    print(e)
            
            

Unique for LLAMA, wanda 25.0%Pruned 1:  6.231003039513678
Unique for LLAMA, wanda 25.0%Pruned 2:  6.055646481178396
Unique for LLAMA, wanda 25.0%Pruned 3:  8.242556281771968
Unique for LLAMA, wanda 43.75%Pruned 1:  8.229166666666666
Unique for LLAMA, wanda 43.75%Pruned 2:  7.9005770084332
Unique for LLAMA, wanda 43.75%Pruned 3:  9.904013961605585
Unique for LLAMA, wanda 57.812%Pruned 1:  8.457997698504027
Unique for LLAMA, wanda 57.812%Pruned 2:  8.791832104367556
Unique for LLAMA, wanda 57.812%Pruned 3:  10.399590163934427
Unique for LLAMA, wanda 68.359%Pruned 1:  9.577677224736048
Unique for LLAMA, wanda 68.359%Pruned 2:  9.090909090909092
Unique for LLAMA, wanda 68.359%Pruned 3:  11.328388401888065
Unique for LLAMA, wanda 76.27%Pruned 1:  10.146103896103897
Unique for LLAMA, wanda 76.27%Pruned 2:  9.536307961504813
Unique for LLAMA, wanda 76.27%Pruned 3:  9.937402190923319
Unique for LLAMA, wanda 0.0%Pruned 1:  5.644355644355644
Unique for LLAMA, wanda 0.0%Pruned 2:  5.7214976861590

In [4]:
base={'pre:tag:nn': 547.0, 'oth:overlap:overlap25': 303.0, 'hyp:tag:nn': 287.0, 'oth:overlap:overlap50': 204.0, 'pre:tag:jj': 161.0, 'pre:tag:.': 158.0, 'hyp:tok:outside': 158.0, 'hyp:tag:dt': 155.0, 'hyp:tok:sleeping': 151.0, 'hyp:tok:for': 140.0, 'hyp:tag:in': 127.0, 'pre:tag:in': 118.0, 'pre:tok:man': 117.0, 'hyp:tok:to': 110.0, 'hyp:tag:vbg': 104.0, 'oth:overlap:overlap75': 99.0, 'pre:tag:dt': 98.0, 'hyp:tok:people': 96.0, 'hyp:tok:there': 96.0, 'hyp:tok:eating': 89.0, 'hyp:tok:outdoors': 83.0, 'pre:tok:woman': 83.0, 'hyp:tok:tall': 80.0, 'hyp:tok:swimming': 79.0, 'hyp:tok:woman': 77.0, 'hyp:tok:sitting': 75.0, 'hyp:tag:prp$': 69.0, 'hyp:tag:ex': 63.0, 'hyp:tok:man': 62.0, 'pre:tok:sitting': 62.0, 'hyp:tag:vb': 57.0, 'pre:tok:girl': 57.0, 'hyp:tag:jj': 57.0, 'hyp:tok:inside': 55.0, 'pre:tok:dog': 52.0, 'hyp:tag:.': 49.0, 'pre:tok:men': 48.0, 'pre:tok:black': 47.0, 'pre:tok:blue': 47.0, 'hyp:tok:men': 46.0, 'pre:tok:people': 45.0, 'hyp:tok:human': 45.0, 'hyp:tok:wearing': 41.0, 'hyp:tok:cat': 39.0, 'pre:tok:walking': 39.0, 'hyp:tok:running': 36.0, 'hyp:tok:playing': 36.0, 'hyp:tok:girl': 34.0, 'hyp:tok:alone': 33.0, 'hyp:tag:nns': 33.0, 'hyp:tok:person': 32.0, 'hyp:tok:walking': 32.0, 'hyp:tok:women': 31.0, 'pre:tok:boy': 31.0, 'hyp:tok:nobody': 30.0, 'hyp:tag:nnp': 30.0, 'pre:tag:vbg': 27.0, 'pre:tok:pool': 27.0, 'pre:tok:women': 27.0, 'hyp:tok:watching': 26.0, 'hyp:tag:prp': 25.0, 'pre:tok:street': 25.0, 'hyp:tok:football': 25.0, 'hyp:tok:dog': 25.0, 'pre:tag:nns': 25.0, 'pre:tok:white': 24.0, 'pre:tok:girls': 24.0, 'hyp:tok:at': 24.0, 'hyp:tok:sits': 22.0, 'hyp:tag:cd': 22.0, 'pre:tok:standing': 21.0, 'hyp:tok:standing': 20.0, 'pre:tok:red': 20.0, 'hyp:tok:about': 20.0, 'hyp:tok:indoors': 19.0, 'hyp:tok:park': 19.0, 'pre:tag:cd': 19.0, 'hyp:tok:in': 19.0, 'hyp:tok:asleep': 19.0, 'hyp:tok:sad': 18.0, 'hyp:tok:black': 18.0, 'hyp:tok:his': 18.0, 'pre:tag:,': 18.0, 'pre:tok:bike': 17.0, 'pre:tag:vbz': 17.0, 'hyp:tok:bike': 17.0, 'hyp:tok:boy': 17.0, 'pre:tok:smiling': 17.0, 'pre:tag:vb': 17.0, 'pre:tok:baseball': 16.0, 'hyp:tok:her': 15.0, 'pre:tok:his': 15.0, 'hyp:tok:water': 15.0, 'pre:tok:ball': 15.0, 'pre:tok:soccer': 14.0, 'pre:tag:cc': 14.0, 'hyp:tok:driving': 14.0, 'pre:tok:running': 14.0, 'pre:tok:and': 14.0, 'hyp:tok:blue': 14.0, 'hyp:tok:something': 14.0, 'hyp:tag:vbz': 13.0, 'pre:tag:prp$': 13.0, 'pre:tok:playing': 13.0, 'pre:tok:green': 13.0, 'hyp:tag:vbp': 13.0, 'pre:tok:riding': 13.0, 'pre:tok:dogs': 13.0, 'pre:tok:water': 13.0, 'hyp:tok:are': 12.0, 'pre:tag:nnp': 12.0, 'pre:tok:for': 12.0, 'pre:tok:snow': 12.0, 'pre:tok:in': 12.0, 'hyp:tok:riding': 12.0, 'pre:tok:beach': 12.0, 'hyp:tok:naked': 12.0, 'hyp:tok:dogs': 12.0, 'hyp:tok:near': 12.0, 'pre:tok:young': 11.0, 'pre:tok:her': 11.0, 'pre:tok:food': 11.0, 'pre:tok:two': 11.0, 'pre:tok:air': 11.0, 'pre:tok:at': 10.0, 'pre:tok:park': 10.0, 'pre:tok:to': 10.0, 'pre:tok:bicycle': 10.0, 'hyp:tok:couch': 10.0, 'hyp:tok:has': 10.0, 'hyp:tok:red': 10.0, 'hyp:tok:after': 10.0, 'hyp:tok:beach': 10.0, 'pre:tok:pink': 10.0, 'hyp:tok:white': 9.0, 'hyp:tok:sleeps': 9.0, 'pre:tok:brown': 9.0, 'pre:tok:wearing': 9.0, 'pre:tok:children': 9.0, 'hyp:tok:is': 9.0, 'pre:tok:on': 9.0, 'pre:tok:field': 8.0, 'hyp:tok:soccer': 8.0, 'pre:tok:table': 8.0, 'pre:tok:sits': 8.0, 'pre:tok:performing': 8.0, 'hyp:tok:two': 8.0, 'pre:tok:while': 7.0, 'pre:tok:something': 7.0, 'hyp:tok:shirt': 7.0, 'pre:tok:blond': 7.0, 'pre:tag:vbp': 7.0, 'hyp:tok:party': 7.0, 'pre:tok:baby': 7.0, 'hyp:tok:talking': 7.0, 'pre:tok:bench': 7.0, 'hyp:tok:friends': 7.0, 'pre:tok:talking': 7.0, 'pre:tok:working': 7.0, 'hyp:tok:home': 7.0, 'hyp:tok:dancing': 7.0, 'pre:tok:race': 6.0, 'pre:tok:outside': 6.0, 'pre:tok:game': 6.0, 'hyp:tok:their': 6.0, 'pre:tok:boat': 6.0, 'pre:tok:jumping': 6.0, 'pre:tok:band': 6.0, 'pre:tok:shirt': 6.0, 'hyp:tok:lady': 6.0, 'pre:tok:yellow': 6.0, 'hyp:tok:girls': 6.0, 'hyp:tok:on': 6.0, 'hyp:tok:race': 6.0, 'hyp:tok:smiling': 5.0, 'hyp:tok:nap': 5.0, 'pre:tok:microphone': 5.0, 'pre:tok:boys': 5.0, 'hyp:tok:baseball': 5.0, 'pre:tok:horse': 5.0, 'hyp:tok:holding': 5.0, 'pre:tok:large': 5.0, 'hyp:tok:dinner': 5.0, 'hyp:tok:wet': 5.0, 'hyp:tok:funny': 5.0, 'pre:tok:person': 5.0, 'hyp:tok:competition': 5.0, 'pre:tok:stage': 5.0, 'hyp:tok:old': 5.0, 'pre:tok:river': 4.0, 'pre:tok:holding': 4.0, 'pre:tok:one': 4.0, 'pre:tok:walks': 4.0, 'hyp:tok:moving': 4.0, 'hyp:tok:while': 4.0, 'hyp:tag:rb': 4.0, 'pre:tok:train': 4.0, 'pre:tok:three': 4.0, 'pre:tag:rb': 4.0, 'hyp:tok:not': 4.0, 'pre:tok:car': 4.0, 'hyp:tok:eats': 4.0, 'pre:tok:as': 4.0, 'hyp:tok:pool': 4.0, 'hyp:tok:game': 4.0, 'hyp:tok:they': 4.0, 'pre:tok:football': 4.0, 'pre:tag:vbn': 4.0, 'pre:tag:prp': 3.0, 'pre:tok:shorts': 3.0, 'pre:tok:city': 3.0, 'pre:tok:grass': 3.0, 'hyp:tok:lunch': 3.0, 'hyp:tok:waiting': 3.0, 'pre:tok:dirt': 3.0, 'pre:tok:ocean': 3.0, 'pre:tok:just': 3.0, 'pre:tok:room': 3.0, 'pre:tok:sand': 3.0, 'hyp:tok:no': 3.0, 'pre:tok:mountain': 3.0, 'pre:tok:old': 3.0, 'pre:tok:dress': 3.0, 'pre:tok:lady': 3.0, 'pre:tok:book': 3.0, 'hyp:tag:vbn': 3.0, 'pre:tok:child': 3.0, 'hyp:tok:and': 3.0, 'hyp:tok:train': 3.0, 'pre:tok:dancing': 3.0, 'pre:tok:with': 3.0, 'pre:tok:walk': 3.0, 'hyp:tok:bed': 3.0, 'pre:tok:construction': 3.0, 'pre:tok:crowd': 3.0, 'pre:tok:other': 3.0, 'pre:tok:camera': 2.0, 'pre:tok:through': 2.0, 'pre:tok:down': 2.0, 'pre:tok:older': 2.0, 'hyp:tok:bikes': 2.0, 'pre:tok:from': 2.0, 'hyp:tok:concert': 2.0, 'pre:tok:group': 2.0, 'pre:tok:climbing': 2.0, 'hyp:tok:tv': 2.0, 'hyp:tok:car': 2.0, 'pre:tok:orange': 2.0, 'hyp:tok:day': 2.0, 'pre:tok:waiting': 2.0, 'hyp:tok:couple': 2.0, 'pre:tok:jumps': 2.0, 'pre:tok:into': 2.0, 'hyp:tok:laying': 2.0, 'pre:tok:are': 2.0, 'hyp:tok:boat': 2.0, 'hyp:tag:cc': 2.0, 'pre:tok:little': 2.0, 'pre:tok:runs': 2.0, 'pre:tok:hat': 2.0, 'hyp:tok:fishing': 2.0, 'pre:tok:guitar': 2.0, 'pre:tok:market': 2.0, 'hyp:tok:field': 2.0, 'pre:tok:cooking': 1.0, 'pre:tok:desk': 1.0, 'pre:tok:kitchen': 1.0, 'hyp:tok:painting': 1.0, 'hyp:tok:holds': 1.0, 'pre:tok:front': 1.0, 'hyp:tok:child': 1.0, 'pre:tok:singing': 1.0, 'hyp:tok:green': 1.0, 'hyp:tok:young': 1.0, 'hyp:tok:stage': 1.0, 'pre:tok:their': 1.0, 'hyp:tok:vacation': 1.0, 'hyp:tok:working': 1.0, 'pre:tok:basketball': 1.0, 'hyp:tok:with': 1.0, 'hyp:tok:baby': 1.0, 'pre:tok:racing': 1.0, 'hyp:tok:cooking': 1.0, 'pre:tok:is': 1.0, 'pre:tok:over': 1.0, 'hyp:tok:skateboarder': 1.0}
found={'pre:tok:riding', 'pre:tok:game', 'pre:tok:to', 'pre:tok:store', 'hyp:tok:nap', 'hyp:tok:are', 'pre:tok:band', 'pre:tag:prp', 'pre:tok:table', 'pre:tok:bike', 'pre:tok:person', 'pre:tok:race', 'pre:tok:boat', 'pre:tok:reading', 'pre:tok:old', 'hyp:tok:riding', 'pre:tok:pool', 'hyp:tok:wearing', 'hyp:tok:sits', 'pre:tok:room', 'hyp:tag:vbp', 'pre:tag:prp$', 'pre:tok:her', 'hyp:tok:dog', 'pre:tok:beach', 'pre:tok:walking', 'pre:tok:his', 'pre:tok:laying', 'hyp:tok:cooking', 'pre:tok:running', 'pre:tok:train', 'pre:tok:horse', 'pre:tag:in', 'pre:tok:performing', 'pre:tok:sits', 'hyp:tag:prp$', 'pre:tok:boys', 'pre:tok:red', 'pre:tok:young', 'pre:tok:microphone', 'pre:tok:swimming', 'pre:tok:while', 'hyp:tag:ex', 'pre:tok:white', 'hyp:tok:cat', 'hyp:tok:red', 'hyp:tok:room', 'pre:tag:nn', 'pre:tok:walks', 'pre:tag:,', 'hyp:tok:in', 'hyp:tok:two', 'hyp:tok:something', 'pre:tok:holding', 'pre:tok:snowboarder', 'hyp:tok:playing', 'hyp:tok:has', 'hyp:tok:person', 'hyp:tok:standing', 'hyp:tok:walks', 'hyp:tag:cd', 'pre:tok:water', 'hyp:tok:beach', 'hyp:tok:is', 'pre:tok:crowd', 'hyp:tok:their', 'pre:tok:sleeping', 'pre:tok:night', 'hyp:tok:funny', 'hyp:tok:near', 'hyp:tok:after', 'pre:tok:jumping', 'hyp:tok:women', 'pre:tok:walk', 'hyp:tok:driving', 'hyp:tok:indoors', 'pre:tok:brown', 'hyp:tok:going', 'pre:tok:guitar', 'pre:tok:dog', 'pre:tok:girl', 'pre:tok:on', 'pre:tag:vbn', 'pre:tok:book', 'pre:tag:cd', 'hyp:tok:sleeping', 'hyp:tok:smiling', 'hyp:tok:old', 'pre:tok:kitchen', 'oth:overlap:overlap50', 'pre:tok:black', 'pre:tok:outside', 'hyp:tok:eating', 'hyp:tok:and', 'hyp:tok:party', 'pre:tok:wearing', 'hyp:tag:nns', 'hyp:tag:vb', 'pre:tag:dt', 'oth:overlap:overlap25', 'pre:tok:cooking', 'pre:tok:baseball', 'pre:tok:through', 'pre:tok:bench', 'hyp:tok:naked', 'pre:tok:chef', 'pre:tok:playing', 'hyp:tok:alone', 'hyp:tok:watching', 'pre:tok:at', 'pre:tok:tennis', 'hyp:tok:at', 'pre:tok:talking', 'pre:tok:football', 'pre:tag:nns', 'hyp:tok:tall', 'pre:tok:is', 'pre:tok:blue', 'hyp:tok:girls', 'pre:tok:working', 'hyp:tag:.', 'hyp:tok:woman', 'hyp:tok:nobody', 'hyp:tok:they', 'hyp:tag:nnp', 'hyp:tok:boys', 'hyp:tok:park', 'hyp:tok:man', 'pre:tok:player', 'pre:tok:snow', 'pre:tok:sitting', 'hyp:tok:to', 'pre:tok:street', 'pre:tok:floor', 'hyp:tok:there', 'hyp:tag:prp', 'hyp:tok:sitting', 'hyp:tok:dancing', 'hyp:tok:people', 'pre:tok:field', 'pre:tok:dancing', 'pre:tok:older', 'pre:tok:something', 'pre:tag:jj', 'hyp:tok:sad', 'hyp:tok:child', 'pre:tok:soccer', 'hyp:tok:friends', 'hyp:tok:boat', 'hyp:tok:water', 'hyp:tok:outdoors', 'hyp:tok:on', 'hyp:tok:race', 'hyp:tag:dt', 'hyp:tok:black', 'pre:tok:river', 'hyp:tok:bed', 'pre:tok:with', 'pre:tok:air', 'pre:tok:and', 'hyp:tag:vbz', 'pre:tok:skateboard', 'pre:tok:mountain', 'pre:tok:wave', 'hyp:tok:about', 'hyp:tok:couch', 'pre:tok:skateboarder', 'hyp:tok:bike', 'hyp:tag:nn', 'hyp:tok:his', 'oth:overlap:overlap75', 'pre:tok:boy', 'pre:tok:food', 'pre:tok:snowy', 'pre:tag:vbg', 'pre:tok:sit', 'pre:tok:runs', 'pre:tag:vb', 'hyp:tok:sleeps', 'hyp:tag:jj', 'pre:tok:women', 'hyp:tok:basketball', 'pre:tok:grass', 'pre:tok:smiling', 'pre:tok:basketball', 'pre:tok:ball', 'hyp:tok:outside', 'hyp:tok:boy', 'hyp:tag:rb', 'pre:tok:eating', 'hyp:tag:in', 'hyp:tok:human', 'hyp:tok:lady', 'hyp:tok:walk', 'hyp:tok:fishing', 'pre:tag:cc', 'pre:tok:park', 'pre:tok:standing', 'pre:tok:ocean', 'pre:tok:hair', 'pre:tok:desk', 'hyp:tok:empty', 'pre:tok:men', 'hyp:tok:with', 'hyp:tok:asleep', 'pre:tok:are', 'pre:tag:vbp', 'pre:tag:.', 'hyp:tok:walking', 'hyp:tok:inside', 'pre:tok:bed', 'hyp:tok:men', 'hyp:tok:swimming', 'hyp:tok:home', 'hyp:tok:no', 'pre:tok:microscope', 'pre:tok:girls', 'pre:tok:down', 'hyp:tok:girl', 'hyp:tag:vbd', 'pre:tok:lady', 'pre:tok:man', 'pre:tok:people', 'hyp:tag:vbg', 'hyp:tok:competition', 'hyp:tok:for', 'pre:tok:stage', 'pre:tok:in', 'hyp:tok:snow', 'pre:tok:dogs', 'hyp:tok:her', 'hyp:tok:blue', 'pre:tag:vbz', 'hyp:tok:young', 'pre:tok:children', 'pre:tok:bicycle', 'pre:tok:woman', 'pre:tok:singing', 'pre:tok:dance', 'hyp:tok:running', 'pre:tok:shirt', 'hyp:tok:chasing', 'pre:tok:car', 'hyp:tok:someone', 'pre:tok:for', 'pre:tok:restaurant'}


In [13]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict,Counter

# NOTE: get_all_cps_for_pi, build_binary_mask must be importable/defined in your env


def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', str(formula))
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts


def get_pruned_folders_sorted(expls_dir):
    """
    Scan expls_dir for *%Pruned folders, return sorted by sparsity ascending.
    Returns list of (sparsity_pct, folder_path).
    """
    folders = []
    for name in os.listdir(expls_dir):
        match = re.match(r'([0-9.]+)%Pruned', name)
        if match and Path(os.path.join(expls_dir, name)).is_dir():
            folders.append((float(match.group(1)), os.path.join(expls_dir, name)))
    folders.sort(key=lambda x: x[0])
    return folders


def get_all_concepts_in_folder(folder_path):
    """
    Collect all concepts from all IOUS1024N CSVs in a folder.
    Matches original get_foundationals logic: extend (not set) per formula.
    """
    concepts = []
    for csv_name in os.listdir(folder_path):
        if 'IOUS1024N' not in csv_name:
            continue
        try:
            df = pd.read_csv(os.path.join(folder_path, csv_name))
            for formula in df.best_name:
                concepts.extend(get_indiv_concepts(formula))
        except Exception as e:
            print(f"  Warning: could not read {csv_name}: {e}")
    return concepts


def get_neuron_concept_freqs(folder_path, concept_list, glob):
    """
    Compute per-concept neuron frequencies using the same pipeline as the
    original code:
        get_all_cps_for_pi  →  build_binary_mask  →  mask.sum(dim=0)

    col_sums[i] = number of neurons that express concept_list[i].
    """
    if not glob:
        concepts = []
        for i in range(1,4):
            concepts_tmp=[]
            df = pd.read_csv(os.path.join(folder_path, f'Cluster{i}IOUS1024N.csv'))
            for formula in df.best_name:
                concepts_tmp.extend(get_indiv_concepts(formula))
            concepts.append(concepts_tmp)
        return concepts
    neuron_formulas = get_all_cps_for_pi(folder_path)
    mask = build_binary_mask(neuron_formulas, concept_list)
    col_sums = mask.sum(dim=0).numpy()

    freqs = {concept_list[i]: float(col_sums[i]) for i in range(len(concept_list))}
    return freqs


def build_folder_list_for_run(expls_dir, baseline_expls_dir):
    """
    Build the sorted folder list for a run, but always substitute the
    0.0%Pruned folder from baseline_expls_dir (the _5 run).
    All other sparsity levels come from expls_dir's own folders.
    """
    folders = get_pruned_folders_sorted(expls_dir)
    if not folders:
        return []

    # Find 0.0%Pruned in the baseline dir
    baseline_zero = None
    for pct, path in get_pruned_folders_sorted(baseline_expls_dir):
        if pct == 0.0:
            baseline_zero = (pct, path)
            break

    if baseline_zero is None:
        print(f"  Warning: no 0.0%Pruned folder in baseline {baseline_expls_dir}, using run's own.")
        return folders

    result = []
    replaced = False
    for pct, path in folders:
        if pct == 0.0:
            result.append(baseline_zero)  # always use _5's dense baseline
            replaced = True
        else:
            result.append((pct, path))

    if not replaced:
        result = [baseline_zero] + result  # run had no 0.0%Pruned, prepend baseline's

    return result


def get_foundationals_for_run(expls_dir, baseline_expls_dir):
    """
    Concepts present in every %Pruned folder for a single run.
    The 0.0%Pruned folder always comes from baseline_expls_dir.
    Matches original get_foundationals logic exactly.
    """
    folders = build_folder_list_for_run(expls_dir, baseline_expls_dir)
    if not folders:
        return set(), []

    per_folder = []
    for _, folder_path in folders:
        concepts = get_all_concepts_in_folder(folder_path)
        if concepts:
            per_folder.append(set(concepts))

    if not per_folder:
        return set(), folders

    return set.intersection(*per_folder), folders


def analyze_foundationals_across_runs(run_expls_dirs, baseline_expls_dir, glob=True):
    valid_dirs = [d for d in run_expls_dirs if Path(d).exists()]
    if not valid_dirs:
        print("No valid run directories found.")
        return {}, []



    # ── Step 1: foundationals = intersection across all runs ──────────────
    
    run_folders = []
    foundationals=[]
    for run_dir in valid_dirs:
        all_foundationals = []
        f, folders = get_foundationals_for_run(run_dir, run_dir)
        if f:
            all_foundationals.append(f)
            run_folders.append(folders)
            
        else:
            print(f"  Warning: no foundational concepts for {run_dir}, skipping.")
        foundational = list(set.intersection(*all_foundationals))
        foundationals.append(foundational)

    if not foundationals:
        print("No foundational concepts found.")
        return {}, []

    # ── Step 2: average freqs across runs per positional level ────────────
    n_levels = max(len(f) for f in run_folders)
    results = {}

    for level_idx in range(n_levels):
        sparsity_idx = level_idx + 1  # 1-based

        run_freqs_local = []
        run_freqs_global = []
        pct_per_run = []

        for run_i, folders in enumerate(run_folders):
            if level_idx >= len(folders):
                print(f"  Run {run_i+1} missing level {sparsity_idx}, skipping.")
                continue

            sparsity_pct, folder_path = folders[level_idx]
            pct_per_run.append(sparsity_pct)

            try:
                freqs_local = get_neuron_concept_freqs(folder_path, foundationals[run_i], False) #freqs should be [c1f, c2f, c3f]
               
                run_freqs_local.append(freqs_local) 
        
                freqs_global = get_neuron_concept_freqs(folder_path, foundationals[run_i], True) #freqs should be [c1f, c2f, c3f]
               
                run_freqs_global.append(freqs_global) 
                    
       
                
            except Exception as e:
                print(f"  Error computing freqs for {folder_path}: {e}")
        
        if not run_freqs_local or not run_freqs_global : #at the end #[(_5 0% freqs c1,c2,c3), (_6 0% freqs)]
            
            continue
        avg=0
        avgs=defaultdict(float)
        for i in run_freqs_global:
            freqs=Counter(i)
            avg += (sum(list(freqs.values()))/len(freqs))
        for i in run_freqs_local:
            for cluster_num, freqs_for_cluster in enumerate(i):
                freqs = Counter(freqs_for_cluster)
                avgs[cluster_num] += (sum(list(freqs.values()))/len(freqs))
                
        for cluster_num in avgs:
            avgs[cluster_num] /= len(run_freqs_local)        
        avg /= len(run_freqs_global)
        #for each cluster avgs[cluster]/=len(run_freqs)
        
        # sort descending by avg freq
        #avg_freqs = dict(sorted(avg_freqs.items(), key=lambda x: x[1], reverse=True))
        results[sparsity_idx] = {
            'pct_per_run':   pct_per_run,
            'n_runs':        len(run_freqs_local),
            'local_avg_mean_freq': avgs,
            'global_avg_mean_freq': avg,
        }

    return results, foundationals


def print_foundational_results(results):
    """
    Table: rows = sparsity levels, cols = C1, C2, C3 (local) + Global
    """
    sorted_keys = sorted(results.keys())

    # figure out number of clusters from first result
    n_clusters = len(results[sorted_keys[0]]['local_avg_mean_freq'])
    cluster_headers = [f"C{i+1}" for i in range(n_clusters)]
    col_w = 10

    # header
    header = f"{'Iter':<8} {'Actual %':<12}" + "".join(f"{h:>{col_w}}" for h in cluster_headers) + f"{'Global':>{col_w}}"
    print("\n" + "=" * len(header))
    print(header)
    print("-" * len(header))

    for sparsity_idx in sorted_keys:
        r = results[sparsity_idx]
        pct_str = '/'.join(f'{p:.2f}' for p in r['pct_per_run'])

        local_vals = "".join(f"{v:>{col_w}.2f}" for v in list(r['local_avg_mean_freq'].values()))
        glob_val = f"{r['global_avg_mean_freq']:>{col_w}.2f}"
        print(f"{sparsity_idx:<8} {pct_str:<12}{local_vals}{glob_val}")

    print("=" * len(header))


# ── Entry point ───────────────────────────────────────────────────────────

if __name__ == "__main__":
    for model in ['LLAMA', 'BERT']:
        for method in ['wanda', 'lottery_ticket', 'CoFi']:
            print(f"AVERAGE FOR {model} {method}")
            runs   = ['Run0.25_5']
            base   = f"/workspace/CCE_NLI/{model.upper()}/exp/{method}/{{run}}/Expls"

            run_expls_dirs     = [base.format(run=run) for run in runs]
            baseline_expls_dir = base.format(run='Run0.25_5')  # 0.0%Pruned always from _5
            results, foundationals = analyze_foundationals_across_runs(
                run_expls_dirs=run_expls_dirs,
                baseline_expls_dir=baseline_expls_dir, glob=False
            )
            if results:
                print_foundational_results(results)
'''
bert
    - cofi redundancy decreases globally and clusterwise
    - lth: redundance increases then decreases (up until 57% then decrease) globally  , 1st cluster increases 2nd3rd decrease
    
llama:
    - lth:increases globally (until 57% then decreases), 1st cluster increases 2nd3rd decrease
    
    wanda red decreases massively
    wanda's foundational concept dif from lth

'''

AVERAGE FOR LLAMA wanda

Iter     Actual %            C1        C2        C3    Global
-------------------------------------------------------------
1        0.00             12.13     13.01      8.50     32.74
2        25.00            11.35     11.86      8.72     32.19
3        43.75             8.86      8.98      7.23     28.48
4        57.81             8.48      8.09      7.15     24.37
5        68.36             7.80      8.06      6.42     19.42
6        76.27             6.88      7.52      7.43     17.16
AVERAGE FOR LLAMA lottery_ticket
  Error computing freqs for /workspace/CCE_NLI/LLAMA/exp/lottery_ticket/Run0.25_5/Expls/76.27%Pruned: [Errno 2] No such file or directory: '/workspace/CCE_NLI/LLAMA/exp/lottery_ticket/Run0.25_5/Expls/76.27%Pruned/Cluster2IOUS1024N.csv'

Iter     Actual %            C1        C2        C3    Global
-------------------------------------------------------------
1        0.00             12.13     13.01      8.50     43.98
2        25.00         

"\nbert\n    - cofi redundancy decreases globally and clusterwise\n    - lth: redundance increases then decreases (up until 57% then decrease) globally  , 1st cluster increases 2nd3rd decrease\n    \nllama:\n    - lth:increases globally (until 57% then decreases), 1st cluster increases 2nd3rd decrease\n    \n    wanda red decreases massively\n    wanda's foundational concept dif from lth\n\n"

In [11]:
if __name__ == "__main__":
    for model in ['LLAMA']:
        for method in ['lottery_ticket']:
            print(f"AVERAGE FOR {model} {method}")
            runs   = ['Run0.25_5']
            base   = f"/workspace/CCE_NLI/{model.upper()}/exp/{method}/{{run}}/Expls"

            run_expls_dirs     = [base.format(run=run) for run in runs]
            baseline_expls_dir = base.format(run='Run0.25_5')  # 0.0%Pruned always from _5
            results, foundationals = analyze_foundationals_across_runs(
                run_expls_dirs=run_expls_dirs,
                baseline_expls_dir=baseline_expls_dir, glob=False
            )
            print(sorted(foundationals))
            if results:
                print_foundational_results(results)

AVERAGE FOR LLAMA lottery_ticket
  Error computing freqs for /workspace/CCE_NLI/LLAMA/exp/lottery_ticket/Run0.25_5/Expls/76.27%Pruned: [Errno 2] No such file or directory: '/workspace/CCE_NLI/LLAMA/exp/lottery_ticket/Run0.25_5/Expls/76.27%Pruned/Cluster2IOUS1024N.csv'
[['pre:tok:for', 'pre:tag:nn', 'pre:tok:boy', 'hyp:tag:.', 'hyp:tag:vb', 'pre:tag:prp', 'pre:tok:is', 'oth:overlap:overlap25', 'pre:tok:her', 'hyp:tok:red', 'pre:tok:men', 'pre:tok:while', 'pre:tok:on', 'pre:tok:at', 'pre:tag:jj', 'pre:tag:prp$', 'pre:tok:street', 'pre:tok:red', 'hyp:tok:is', 'pre:tok:and', 'hyp:tok:their', 'pre:tok:his', 'hyp:tok:on', 'pre:tok:floor', 'hyp:tok:young', 'hyp:tok:there', 'pre:tok:down', 'hyp:tok:for', 'pre:tag:cc', 'hyp:tok:with', 'hyp:tok:boys', 'pre:tag:vbp', 'hyp:tag:cd', 'hyp:tag:nns', 'hyp:tok:has', 'pre:tok:blue', 'hyp:tok:sitting', 'pre:tok:people', 'hyp:tok:man', 'hyp:tok:and', 'hyp:tok:dog', 'pre:tok:wearing', 'hyp:tok:child', 'pre:tag:vbg', 'hyp:tok:girl', 'hyp:tag:jj', 'hyp:tag:i

In [204]:
avg mean freq: 24.42 25.47
avg mean freq: 24.29
avg mean freq: 24.35
avg mean freq: 24.72
avg mean freq: 24.82
avg mean freq: 24.67

SyntaxError: invalid syntax (2466716767.py, line 1)

In [190]:
def get_foundationals(root_dir):
    root_path = Path(root_dir)
    # Find all matching CSV files
    fldr_pattern = '*%Pruned'
    fldr_files = list(root_path.rglob(fldr_pattern))
    cross_iter_concept_dict=defaultdict(list)
    for fldr_file in fldr_files:
        concepts = []
        for csvs in os.listdir(os.path.join(root_dir, fldr_file)):
            if 'IOUS1024N' not in csvs: continue
            csv_file = os.path.join(root_dir, fldr_file, csvs)
            df = pd.read_csv(csv_file)
            for unit, formula in zip(df.unit, df.best_name):
                concepts.extend(get_indiv_concepts(formula))
        cross_iter_concept_dict[fldr_file]=set(concepts)
    preserved_concepts = set.intersection(*cross_iter_concept_dict.values())
    return list(preserved_concepts)
def get_topk_concepts(mask,k, concept_list):
    """Test if concepts are uniformly distributed across neurons."""
    col_sums = mask.sum(dim=0).numpy()
    sorted_idx = np.argsort(col_sums)[::-1]
    top=[]
    freqs={}
    
    for i in range(len(sorted_idx)):
        idx = sorted_idx[i]
        top.append(concept_list[idx])
        freqs[concept_list[idx]]=col_sums[idx]
    return top, sum(list(freqs.values()))/len(freqs), freqs
root_dir='/workspace/CCE_NLI/LLAMA/exp/lottery_ticket/Run0.25_5/Expls'
foundationals = get_foundationals(root_dir)
for direct in sorted(os.listdir(root_dir)):
    neuron_formulas = get_all_cps_for_pi(f'/workspace/CCE_NLI/LLAMA/exp/lottery_ticket/Run0.25_5/Expls/{direct}')
    mask = build_binary_mask(neuron_formulas, foundationals)
    top,f,freqs=get_topk_concepts(mask,len(foundationals), foundationals)
    print(direct,f)
    print(freqs)
'''
'pre:tag:nn': 539.0,
 'hyp:tag:nn': 327.0,
 'oth:overlap:overlap25': 307.0,
 'oth:overlap:overlap50': 202.0,
 'pre:tag:jj': 178.0,
 'hyp:tok:outside': 149.0,
 'hyp:tok:sleeping': 145.0,
 'hyp:tag:in': 134.0,
 'hyp:tag:vbg': 133.0,
 'hyp:tag:dt': 133.0,
 'pre:tag:.': 133.0,
 'hyp:tok:for': 131.0,
'''


0.0%Pruned 25.410569105691057
{'pre:tag:nn': 656.0, 'hyp:tag:nn': 242.0, 'hyp:tok:sleeping': 232.0, 'hyp:tok:outside': 164.0, 'pre:tok:man': 150.0, 'oth:overlap:overlap25': 146.0, 'pre:tok:sitting': 141.0, 'pre:tag:.': 137.0, 'hyp:tok:nobody': 135.0, 'hyp:tok:for': 120.0, 'hyp:tok:sitting': 119.0, 'pre:tag:in': 110.0, 'pre:tag:jj': 109.0, 'hyp:tag:in': 108.0, 'pre:tag:dt': 103.0, 'hyp:tok:to': 97.0, 'hyp:tag:dt': 93.0, 'hyp:tok:asleep': 87.0, 'pre:tok:woman': 86.0, 'pre:tok:walking': 82.0, 'hyp:tok:outdoors': 81.0, 'hyp:tag:vbg': 81.0, 'pre:tok:dog': 75.0, 'hyp:tok:man': 72.0, 'pre:tok:men': 71.0, 'hyp:tag:prp$': 70.0, 'hyp:tok:men': 68.0, 'hyp:tok:woman': 66.0, 'hyp:tok:tall': 63.0, 'pre:tok:sits': 61.0, 'hyp:tag:nns': 59.0, 'hyp:tok:walking': 58.0, 'hyp:tok:people': 53.0, 'hyp:tok:person': 48.0, 'hyp:tok:swimming': 47.0, 'pre:tok:girl': 46.0, 'hyp:tok:there': 46.0, 'hyp:tok:eating': 46.0, 'hyp:tag:.': 45.0, 'hyp:tok:alone': 40.0, 'hyp:tag:jj': 39.0, 'pre:tok:boy': 39.0, 'pre:tok:pool

"\n'pre:tag:nn': 539.0,\n 'hyp:tag:nn': 327.0,\n 'oth:overlap:overlap25': 307.0,\n 'oth:overlap:overlap50': 202.0,\n 'pre:tag:jj': 178.0,\n 'hyp:tok:outside': 149.0,\n 'hyp:tok:sleeping': 145.0,\n 'hyp:tag:in': 134.0,\n 'hyp:tag:vbg': 133.0,\n 'hyp:tag:dt': 133.0,\n 'pre:tag:.': 133.0,\n 'hyp:tok:for': 131.0,\n"

In [106]:
from collections import Counter
from itertools import chain

def get_nonfoundational_freq(neuron_formulas, foundationals):
    # flatten all concepts
    all_concepts = chain.from_iterable(neuron_formulas.values())

    # keep only non-foundationals
    non_foundationals = [c for c in all_concepts if c not in foundationals]

    # count frequency
    return Counter(non_foundationals)
from collections import Counter
from itertools import chain

def get_all_concept_freq(neuron_formulas):
    a=[]
    for u, co in neuron_formulas.items():
        for c in co:
            a.append(c)
    
    return Counter(a)

In [107]:
def get_pi(model, method, run):
    """
    Collect sparsity values (the number before '%Pruned')
    from directory names inside:

    /workspace/CCE_NLI/{MODEL}/exp/{method}/{run}/Expls

    Returns:
        Sorted list of floats.
    """
    base_path = f"/workspace/CCE_NLI/{model.upper()}/exp/{method}/{run}/Expls"

    if not os.path.exists(base_path):
        raise FileNotFoundError(f"Path not found: {base_path}")

    sparsities = []

    for name in os.listdir(base_path):
        match = re.match(r"([0-9.]+)%Pruned", name)
        if match:
            sparsities.append(float(match.group(1)))
    if 0.0 not in sparsities:
        sparsities.insert(0,0.0)
    return sorted(sparsities)

run='Run0.25_5'
pis = get_pi('BERT', 'lottery_ticket', run)
print(pis)
for i,pi in enumerate(pis):
    if pi == 0.0:
        s=[]
        neuron_formulas = get_all_cps_for_pi(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned')
        for u in neuron_formulas:
            for i in neuron_formulas[u]:
                s.append(i)
        print("ubique ", len(s))
    else:
        neuron_formulas = get_all_cps_for_pi(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/{run}/Expls/{pi}%Pruned')
        
    mask = build_binary_mask(neuron_formulas, foundationals)
    top,f,_=get_topk_concepts(mask,len(foundationals), foundationals)
    print(f"Avg number of times a foundational concept is repeated at sparsity {pi}%: {f}")
    nonfound=get_nonfoundational_freq(neuron_formulas, foundationals)
    print(f"sum of nonfound frewq : {sum(list(nonfound.values()))}. num found: {len(nonfound)}, percent:{sum(list(nonfound.values()))/len(nonfound)}")
    
    all_freq = get_all_concept_freq(neuron_formulas)
    total = sum(all_freq.values())
    unique = len(all_freq)

    print(
        f"All concepts total freq: {total}, "
        f"unique: {unique}, "
        f"avg repetition: {total / unique if unique else 0}"
    )


[0.0, 25.0, 43.75, 57.812, 68.359, 76.27]
IN  /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster1IOUS1024N.csv
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster1IOUS1024N.csv 2183
IN  /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster2IOUS1024N.csv
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster2IOUS1024N.csv 4295
IN  /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster3IOUS1024N.csv
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster3IOUS1024N.csv 7279
ubique  7279
sum of freqs : 5881.0. num found: 151
Avg number of times a foundational concept is repeated at sparsity 0.0%: 38.94701986754967
sum of nonfound frewq : 139

In [91]:
for i,pi in enumerate(['0.0', '25.0', '43.75', '57.812', '68.359', '76.27']):
    gen=0
    unit, unique1 = load_csv_data(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned/Cluster1IOUS1024N.csv')
    unit, unique2 = load_csv_data(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned/Cluster2IOUS1024N.csv')
    unit, unique3 = load_csv_data(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned/Cluster3IOUS1024N.csv')
    unique=unique1|unique2|unique3
    for cp in unique:
        if any(kw in cp for kw in [':tag:', 'oth:overlap']):
            gen += 1
    print(pi, gen/len(unique), gen, len(unique))

0.0 0.09368635437881874 46 491
25.0 0.09484536082474226 46 485
43.75 0.09270216962524655 47 507
57.812 0.09021113243761997 47 521
68.359 0.09325396825396826 47 504
76.27 0.0874751491053678 44 503


In [66]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/43.75%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.045283018867924

In [74]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/57.812%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.026415094339622

In [75]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/68.359%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.713207547169812

In [76]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/76.27%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.00377358490566

In [71]:
def get_non_foundationals(foundational, pi):
    unit_to_cp_dict = get_all_cps_for_pi(pi)
    allcps = set()
    for unit,cps in unit_to_cp_dict.items():
        allcps.update(cps)
    return list(allcps - set(foundational))
def get_topk_concepts(mask,k, concept_list):
    """Test if concepts are uniformly distributed across neurons."""
    col_sums = mask.sum(dim=0).numpy()
    sorted_idx = np.argsort(col_sums)[::-1]
    top=[]
    freqs={}
    for i in range(len(sorted_idx))[:k]:
        idx = sorted_idx[i]
        top.append(concept_list[idx])
        freqs[concept_list[idx]]=col_sums[idx]
    return top, sum(list(freqs.values()))/len(freqs)
root_dir='/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls'
foundationals = get_foundationals(root_dir)
get_non_foundationals(foundationals,'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/25.0%Pruned' )
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/25.0%Pruned')
mask = build_binary_mask(neuron_formulas, non_foundationals)
_,f=get_topk_concepts(mask,len(non_foundationals), non_foundationals)
f

0.7145390070921985

In [70]:
get_non_foundationals(foundationals,'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/43.75%Pruned' )
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/43.75%Pruned')
mask = build_binary_mask(neuron_formulas, non_foundationals)
_,f=get_topk_concepts(mask,len(non_foundationals), non_foundationals)
f

0.6968085106382979